## Event/site preprocessing
The primary goal of this preprocessing script is to project the expected foot traffic at each proprosed event

In [1]:
import pandas as pd
import numpy as np

propSites = pd.read_csv("../../data/proposedSites.csv")
sites = pd.read_csv("../../data/sites.csv")
events = pd.read_csv("../../data/events.csv")


### Determining expected attenace
There are three types of proposed site visits: 
1. Sites that have already been visited
2. Sites that have not been visited, but are a type (e.g., Library, thirft shop, etc) that has been visited already
3. Sites that have not been visited, and are of a type that has never been visited before. 

To handle each of these cases:

*Case 1*: Calculate statistics from historical data of that *exact* given site

*Case 2*: Calculate statistics from historical data for that *type* of site

*Case 3*: Calculate statistics from all sites ever visited


In [2]:
propSites["minAtt"] = np.ones((propSites.shape[0], 1))*-99
propSites["medAtt"] = np.ones((propSites.shape[0], 1))*-99
propSites["maxAtt"] = np.ones((propSites.shape[0], 1))*-99


for r, row in propSites.iterrows():
    name = row["siteName"]
    city = row["city"]

    # Determine the site ID (if it exists)
    site_mask = np.logical_and(sites.loc[:, "siteName"] == name, sites.loc[:, "city"] == city)

    siteId = sites.loc[site_mask, "siteId"].values
    siteType = row["siteType"].lower()

    if len(siteId) > 1: 
        ValueError("Error: duplicate events found in the site table.")        

    # Check if this site has been visisted yet 
    if len(siteId) == 0: 
        # Place holder if site has not been visited
        siteId = -99 
    else: 
        siteId = siteId[0]

    # Determine the events at the same location 
    same_site_mask = events.loc[:,"siteId"] == siteId
    past_events_same_site = events.loc[same_site_mask, :]

    # Determine the events of the same type 
    event_sites = events.merge(sites, on="siteId").loc[:,["eventId", "siteType", "footTraffic"]].drop_duplicates()
    events_same_type_mask = event_sites.loc[:,"siteType"] == siteType
    events_same_type = event_sites.loc[events_same_type_mask, :]

    #events_same_type_mask =  events.loc[:, "siteId"] 

    # CASE 1: Site has been visited before 
    if past_events_same_site.shape[0] > 0:
        propSites.loc[r, "minAtt"] = past_events_same_site.loc[:,"footTraffic"].min()
        propSites.loc[r, "medAtt"] = np.round(past_events_same_site.loc[:,"footTraffic"].median())
        propSites.loc[r, "maxAtt"] = past_events_same_site.loc[:,"footTraffic"].max()

    # CASE 2: Site not visited, but sites of the same type 
    elif siteId == -99 and events_same_type.shape[0] > 0:
        propSites.loc[r, "minAtt"] = events_same_type.loc[:,"footTraffic"].min()
        propSites.loc[r, "medAtt"] = np.round(events_same_type.loc[:,"footTraffic"].median())
        propSites.loc[r, "maxAtt"] = events_same_type.loc[:,"footTraffic"].max()

    # CASE 3: Site not visited, new site type
    else: 
        propSites.loc[r, "minAtt"] = events.loc[:,"footTraffic"].min()
        propSites.loc[r, "medAtt"] = np.round(events.loc[:,"footTraffic"].median())
        propSites.loc[r, "maxAtt"] = events.loc[:,"footTraffic"].max()

propSites

,siteName,siteType,city,minAtt,medAtt,maxAtt
0,Alger Dinner,charitable meal,Alger,1.0,3.0,5.0
1,Dunkirk Dinner,charitable meal,Dunkirk,2.0,3.0,7.0
2,Lima ODB,charitable meal,Lima,2.0,3.0,4.0
3,Saint Marks UMC,charitable meal,Lima,2.0,3.0,7.0
4,Grace Clinics,clinic,Marion,0.0,3.0,10.0
...,...,...,...,...,...,...
92,Union County Personal Needs Pantry,food pantry,Marysville,0.0,3.0,10.0
93,Plain City Food Pantry,food pantry,Plain City,0.0,3.0,10.0
94,REAP Food Pantry,food pantry,Richwood,0.0,3.0,10.0
95,North Union Personal Needs Pantry,food pantry,Richwood,0.0,3.0,10.0


## Export data

In [3]:
propSites.to_csv("../../data/proposedSitesProcessed.csv")

